# PhysicsGPT From Scratch - Colab Notebook

This notebook trains a small decoder-only GPT model from scratch on physics-style text and QA data.

## 1. Check GPU runtime

In [ ]:
!nvidia-smi || true
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

## 2. Clone or refresh the repository

In [ ]:
%cd /content
!rm -rf physics-gpt-from-scratch
!git clone https://github.com/harkarshaurya-eng/Sara-Phy-Chatbot.git physics-gpt-from-scratch
%cd /content/physics-gpt-from-scratch
!pwd
!ls

## 3. Install requirements

In [ ]:
%cd /content/physics-gpt-from-scratch
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. Mount Google Drive (optional but recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Download datasets

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/download_datasets.py --only-group physics --only-group conversation --max-samples-per-dataset 5000

## 6. Prepare corpus

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/prepare_text_corpus.py

## 7. Train tokenizer from scratch

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/train_tokenizer.py --config configs/tiny_gpt.yaml

## 8. Tokenize dataset

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/tokenize_dataset.py --config configs/tiny_gpt.yaml

## 9. Train tiny GPT model

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/train.py --model_config configs/tiny_gpt.yaml --train_config configs/train_config.yaml

## 10. Resume training if needed

In [ ]:
%cd /content/physics-gpt-from-scratch
# !python src/train.py --model_config configs/tiny_gpt.yaml --train_config configs/train_config.yaml --resume checkpoints/last_checkpoint.pt

## 11. Evaluate the model

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/evaluate.py --checkpoint checkpoints/final_model.pt

## 12. Generate sample text

In [ ]:
%cd /content/physics-gpt-from-scratch
!python src/generate.py --checkpoint checkpoints/final_model.pt --prompt "Explain Newton's second law."

## 13. Launch Gradio chatbot

In [ ]:
%cd /content/physics-gpt-from-scratch
!python app/gradio_chatbot.py --share

## 14. Save checkpoints to Google Drive

In [ ]:
import os, shutil
target = '/content/drive/MyDrive/physics-gpt-checkpoints'
os.makedirs(target, exist_ok=True)
for item in ['checkpoints', 'data/tokenizer']:
    source = f'/content/physics-gpt-from-scratch/{item}'
    destination = os.path.join(target, os.path.basename(item))
    if os.path.isdir(destination):
        shutil.rmtree(destination)
    shutil.copytree(source, destination)
print('Saved checkpoints and tokenizer to', target)